# CitizenTech · Visão Computacional para Manutenção Urbana
**Triagem de fissuras em concreto com transferência de aprendizado (MobileNetV3-Small)**

Este notebook cobre as 5 etapas da atividade:

| # | Etapa | Onde |
|---|-------|------|
| 1 | Dados: base, amostra, pré-processamento, divisão **sem vazamento** entre treino/validação/teste | seções 1–2 |
| 2 | Modelo: CNN pré-treinada + transferência de aprendizado | seção 3 |
| 3 | Avaliação: precisão, recall, F1, matriz de confusão, análise de acertos/erros | seções 4–5 |
| 4 | Integração CitizenTech: módulo Python + app Streamlit + processamento de vídeo | seção 6 |
| 5 | Desempenho: latência, FPS, equipamento, justificativa da arquitetura e do limiar | seções 7–8 |

**Base escolhida:** *Concrete Crack Images for Classification* (Mendeley, 40 mil imagens 227×227, classes `Positive` = com fissura e `Negative` = sem fissura). É a base recomendada para o prazo de 4 horas.

**Como rodar:** Google Colab com GPU (Ambiente de execução → Alterar tipo → T4). Execute as células em ordem; o treino completo leva alguns minutos. Se o download automático da base falhar, a célula da seção 1 explica como subir os arquivos manualmente.

> **Aviso de escopo.** O sistema apoia a **triagem de locais para inspeção**. A presença de fissura na imagem, isoladamente, não permite determinar a gravidade do dano nem a segurança estrutural da edificação.

## 0. Preparação do ambiente

In [ ]:
!pip -q install streamlit ipywidgets

In [ ]:
import os, sys, random, time, json, platform, subprocess, shutil, copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T

from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
                             confusion_matrix, precision_recall_curve, roc_auc_score,
                             classification_report)
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()

IN_COLAB = "google.colab" in sys.modules
WORK = Path(os.environ.get("CT_WORK", "/content" if IN_COLAB else "."))
DATA_DIR = WORK / "data"
OUT_DIR = WORK / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Hiperparâmetros e decisões do experimento (altere aqui e rode de novo para ver o efeito)
CFG = dict(
    n_per_class=3000,     # amostra por classe (de 20 000 disponíveis)
    img_size=224,         # resolução de entrada da CNN
    batch=64,
    epochs_head=3,        # etapa 1: só a cabeça treina (backbone congelado)
    epochs_ft=5,          # etapa 2: ajuste fino de toda a rede
    lr_head=1e-3,
    lr_ft=1e-4,
    n_groups=200,         # nº de "grupos de origem" estimados (ver seção 2)
    test_frac=0.15,
    val_frac=0.15,
    num_workers=2,
    pretrained=True,
    compare_arch=True,    # treina também a ResNet18 só para comparar com a escolhida
)
CFG.update(json.loads(os.environ.get("CT_CFG", "{}")))   # (opcional) sobrescrever por variável de ambiente

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def hw_info():
    cpu = platform.processor() or platform.machine()
    try:
        with open("/proc/cpuinfo") as f:
            for line in f:
                if line.startswith("model name"):
                    cpu = line.split(":", 1)[1].strip(); break
    except Exception:
        pass
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    return cpu, gpu, os.cpu_count()

CPU_NAME, GPU_NAME, N_CPU = hw_info()
print("Dispositivo de treino:", DEVICE, "|", GPU_NAME or CPU_NAME)
print("torch", torch.__version__)

R = {}   # dicionário com todos os resultados, usado no relatório final

def savefig(name):
    plt.savefig(OUT_DIR / name, dpi=130, bbox_inches="tight")

## 1. Dados: obtenção, amostra e inspeção

A base vem de 458 fotos de alta resolução (4032×3024) recortadas em imagens 227×227. Foi coletada em prédios do campus da METU e **não** tem aumento de dados aplicado. Vamos:

1. baixar e localizar as pastas `Positive` / `Negative`;
2. sortear uma **amostra balanceada** (prazo de 4 h);
3. verificar integridade das imagens e olhar exemplos.

In [ ]:
URL = ("https://data.mendeley.com/datasets/5y9wdsg2zt/1/files/"
       "c0d86f9f-852e-4d00-bf45-9a0e24e3b932/Concrete%20Crack%20Images%20for%20Classification.rar")

def find_class_dirs(root):
    pos = neg = None
    for dp, dn, fn in os.walk(root):
        name = Path(dp).name.lower()
        if fn and name == "positive": pos = Path(dp)
        if fn and name == "negative": neg = Path(dp)
    return (pos, neg) if pos and neg else None

def ensure_dataset():
    found = find_class_dirs(DATA_DIR)
    if found:
        return found
    pkg = DATA_DIR / "concrete_cracks.rar"
    try:
        subprocess.run(["wget", "-q", "-O", str(pkg), URL], check=True, timeout=1800)
        subprocess.run("apt-get -qq install -y p7zip-full unrar > /dev/null 2>&1", shell=True)
        r = subprocess.run(["7z", "x", "-y", f"-o{DATA_DIR}", str(pkg)], stdout=subprocess.DEVNULL)
        if r.returncode != 0:
            subprocess.run(["unrar", "x", "-o+", str(pkg), str(DATA_DIR) + "/"], stdout=subprocess.DEVNULL)
    except Exception as e:
        print("Download direto falhou:", e)
    found = find_class_dirs(DATA_DIR)
    if found:
        return found
    try:   # plano B: mesma base espelhada no Kaggle
        import kagglehub
        p = kagglehub.dataset_download("arunrk7/surface-crack-detection")
        found = find_class_dirs(p)
        if found:
            return found
    except Exception as e:
        print("Plano B (kagglehub) falhou:", e)
    raise FileNotFoundError(
        "Não consegui obter a base automaticamente.\n"
        "Baixe em https://data.mendeley.com/datasets/5y9wdsg2zt/1 , extraia e envie as pastas "
        f"'Positive' e 'Negative' para {DATA_DIR}/ (Colab: painel de arquivos à esquerda), depois rode esta célula de novo.")

pos_dir, neg_dir = ensure_dataset()
print("Positive:", pos_dir, "\nNegative:", neg_dir)

def list_imgs(d):
    return sorted(p for p in Path(d).iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))

rng = np.random.RandomState(SEED)
rows, avail = [], {}
for label, d in [(0, neg_dir), (1, pos_dir)]:
    files = list_imgs(d)
    avail[label] = len(files)
    sel = rng.choice(len(files), min(CFG["n_per_class"], len(files)), replace=False)
    rows += [(str(files[i]), label) for i in sel]
df = pd.DataFrame(rows, columns=["path", "label"]).sample(frac=1, random_state=SEED).reset_index(drop=True)

# integridade: remove arquivos corrompidos, se houver
ok = []
for p in df.path:
    try:
        with Image.open(p) as im:
            im.verify()
        ok.append(True)
    except Exception:
        ok.append(False)
df = df[np.array(ok)].reset_index(drop=True)

print(f"Disponível: {avail[0]} sem fissura / {avail[1]} com fissura  ->  amostra: {len(df)} imagens")
print(df.label.value_counts().rename({0: "sem fissura", 1: "com fissura"}))
with Image.open(df.path[0]) as im:
    print("Tamanho original das imagens:", im.size, im.mode)
R["data"] = dict(avail_neg=avail[0], avail_pos=avail[1], sample=len(df))

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for r, lab in enumerate([0, 1]):
    sub = df[df.label == lab].head(6)
    for c, p in enumerate(sub.path):
        axes[r, c].imshow(Image.open(p).convert("RGB")); axes[r, c].axis("off")
    axes[r, 0].set_title("SEM fissura" if lab == 0 else "COM fissura", loc="left", fontsize=11)
plt.suptitle("Exemplos da base (cada linha = uma classe)")
plt.tight_layout(); savefig("fig01_exemplos.png"); plt.show()

## 2. Pré-processamento e divisão **sem vazamento**

**Pré-processamento.** Redimensionamento para 224×224 (entrada padrão das CNNs ImageNet) e normalização com média/desvio do ImageNet, porque o backbone foi pré-treinado assim. No treino usamos aumento de dados leve (giros e variação de brilho/contraste/saturação), que simula mudanças de câmera e iluminação; validação e teste ficam **sem** aumento.

**O problema do vazamento.** A atividade pede para não distribuir recortes da mesma imagem original entre conjuntos diferentes. Nesta base, **os nomes dos arquivos não dizem de qual das 458 fotos cada recorte veio**, então não dá para agrupar por metadado. A solução adotada é estimar os grupos de origem pela aparência:

1. extrair um vetor de características de cada recorte com uma CNN pré-treinada (sem usar o rótulo);
2. agrupar com K-Means (`n_groups` grupos): recortes da mesma foto têm superfície, textura e iluminação muito parecidas e tendem a cair no mesmo grupo;
3. dividir **por grupo** (`GroupShuffleSplit`): um grupo inteiro vai para treino, validação **ou** teste.

É uma aproximação, não uma garantia: pode separar recortes de uma mesma foto em grupos diferentes. Em compensação, deixa o teste mais exigente (superfícies que o modelo não viu), então o resultado tende a ser mais honesto do que um sorteio aleatório de recortes.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
S = CFG["img_size"]

tfm_eval = T.Compose([T.Resize((S, S)), T.ToTensor(), T.Normalize(MEAN, STD)])
tfm_train = T.Compose([T.Resize((S, S)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                       T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
                       T.ToTensor(), T.Normalize(MEAN, STD)])

class ImgDS(Dataset):
    def __init__(self, paths, labels, tfm):
        self.paths, self.labels, self.tfm = list(paths), list(labels), tfm
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tfm(img), torch.tensor(self.labels[i], dtype=torch.float32)

# 1) vetores de características (sem usar rótulos)
emb = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if CFG["pretrained"] else None)
emb.classifier = nn.Identity()
emb.eval().to(DEVICE)
feats = []
with torch.no_grad():
    for x, _ in DataLoader(ImgDS(df.path, df.label, tfm_eval), batch_size=128, num_workers=CFG["num_workers"]):
        feats.append(emb(x.to(DEVICE)).cpu().numpy())
feats = np.concatenate(feats)
feats /= (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-8)
del emb

# 2) grupos de origem estimados
k = max(5, min(CFG["n_groups"], len(df) // 10))
df["group"] = KMeans(n_clusters=k, n_init=3, random_state=SEED).fit_predict(feats)

# 3) divisão por grupo: teste -> validação -> treino
gss = GroupShuffleSplit(n_splits=1, test_size=CFG["test_frac"], random_state=SEED)
trv_idx, te_idx = next(gss.split(df, groups=df.group))
trv = df.iloc[trv_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=CFG["val_frac"] / (1 - CFG["test_frac"]), random_state=SEED)
tr_i, va_i = next(gss2.split(trv, groups=trv.group))
df_tr, df_va, df_te = trv.iloc[tr_i].reset_index(drop=True), trv.iloc[va_i].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)

g_tr, g_va, g_te = set(df_tr.group), set(df_va.group), set(df_te.group)
assert not (g_tr & g_va) and not (g_tr & g_te) and not (g_va & g_te), "vazamento de grupos entre conjuntos!"

summary = pd.DataFrame({
    "imagens": [len(df_tr), len(df_va), len(df_te)],
    "com fissura": [int(d.label.sum()) for d in (df_tr, df_va, df_te)],
    "sem fissura": [int((1 - d.label).sum()) for d in (df_tr, df_va, df_te)],
    "grupos": [len(g_tr), len(g_va), len(g_te)],
}, index=["treino", "validação", "teste"])
summary["% fissura"] = (100 * summary["com fissura"] / summary["imagens"]).round(1)
display(summary) if "display" in globals() else print(summary)
R["split"] = summary.to_dict()

def make_loader(d, tfm, shuffle):
    return DataLoader(ImgDS(d.path, d.label, tfm), batch_size=CFG["batch"], shuffle=shuffle,
                      num_workers=CFG["num_workers"], pin_memory=torch.cuda.is_available())
train_dl = make_loader(df_tr, tfm_train, True)
val_dl = make_loader(df_va, tfm_eval, False)
test_dl = make_loader(df_te, tfm_eval, False)

## 3. Modelo: transferência de aprendizado

**Arquitetura escolhida: MobileNetV3-Small** pré-treinada no ImageNet, com a última camada trocada por 1 neurônio (probabilidade de fissura, via sigmoide). Treino em duas etapas:

1. **Cabeça** (backbone congelado): ensina só o classificador novo, rápido e estável;
2. **Ajuste fino** (tudo liberado, taxa de aprendizado 10× menor): adapta os filtros ao domínio concreto/fissura.

Mantém-se o checkpoint com melhor F1 na **validação** (o teste só é usado no final).

Motivo da escolha (detalhado na seção 8): a tarefa é binária e visualmente simples, e o objetivo é monitoramento contínuo em equipamento modesto, então um modelo leve com pouca perda de qualidade é o melhor equilíbrio. A ResNet18 é treinada como comparação.

In [ ]:
HEAD_NAMES = {"classifier", "fc"}

def build_model(arch, pretrained=True):
    if arch == "mobilenet_v3_small":
        m = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, 1)
    elif arch == "mobilenet_v3_large":
        m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, 1)
    elif arch == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, 1)
    else:
        raise ValueError(arch)
    return m

def set_stage(model, stage):
    """stage 1: só a cabeça treina. stage 2: toda a rede treina."""
    for name, child in model.named_children():
        for p in child.parameters():
            p.requires_grad = (stage == 2) or (name in HEAD_NAMES)

@torch.no_grad()
def predict_probs(model, loader):
    model.eval(); ps, ys = [], []
    for x, y in loader:
        ps.append(torch.sigmoid(model(x.to(DEVICE)).squeeze(1)).cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(ps), np.concatenate(ys).astype(int)

def train_model(arch):
    seed_all()
    model = build_model(arch, CFG["pretrained"]).to(DEVICE)
    crit = nn.BCEWithLogitsLoss()
    hist, best = [], dict(f1=-1.0, vloss=1e9, state=None, epoch=-1)
    t0, ep_global = time.time(), 0
    for stage, epochs, lr in [(1, CFG["epochs_head"], CFG["lr_head"]), (2, CFG["epochs_ft"], CFG["lr_ft"])]:
        set_stage(model, stage)
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
        for _ in range(epochs):
            ep_global += 1
            model.train()
            if stage == 1:   # backbone congelado também não atualiza estatísticas do BatchNorm
                for name, child in model.named_children():
                    if name not in HEAD_NAMES: child.eval()
            tot, n = 0.0, 0
            for x, y in train_dl:
                x, y = x.to(DEVICE), y.to(DEVICE)
                opt.zero_grad()
                loss = crit(model(x).squeeze(1), y)
                loss.backward(); opt.step()
                tot += loss.item() * len(x); n += len(x)
            pv, yv = predict_probs(model, val_dl)
            vloss = float(F.binary_cross_entropy(torch.tensor(pv).clamp(1e-6, 1 - 1e-6), torch.tensor(yv, dtype=torch.float32)))
            vf1 = f1_score(yv, pv >= 0.5, zero_division=0)
            hist.append(dict(epoca=ep_global, etapa=stage, loss_treino=tot / n, loss_val=vloss, f1_val=vf1))
            if vf1 > best["f1"] or (vf1 == best["f1"] and vloss < best["vloss"]):
                best = dict(f1=vf1, vloss=vloss, epoch=ep_global,
                            state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
            print(f"[{arch}] época {ep_global:02d} (etapa {stage}) loss_treino={tot/n:.4f} loss_val={vloss:.4f} F1_val={vf1:.4f}")
    model.load_state_dict(best["state"])
    print(f"[{arch}] melhor época: {best['epoch']} (F1 val = {best['f1']:.4f}) | tempo de treino: {time.time()-t0:.0f}s")
    return model.eval(), pd.DataFrame(hist), time.time() - t0, best["epoch"]

MAIN_ARCH = "mobilenet_v3_small"
model, hist, t_train, best_ep = train_model(MAIN_ARCH)
R["train"] = dict(arch=MAIN_ARCH, seconds=t_train, best_epoch=int(best_ep), epochs=len(hist))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(hist.epoca, hist.loss_treino, marker="o", label="treino"); ax[0].plot(hist.epoca, hist.loss_val, marker="o", label="validação")
ax[0].set_title("Loss por época"); ax[0].set_xlabel("época"); ax[0].legend()
ax[1].plot(hist.epoca, hist.f1_val, marker="o", color="tab:green"); ax[1].set_title("F1 na validação (limiar 0,5)"); ax[1].set_xlabel("época")
for a in ax:
    a.axvline(CFG["epochs_head"] + 0.5, color="gray", ls="--", lw=1)
    a.text(CFG["epochs_head"] + 0.6, a.get_ylim()[0], " ajuste fino", color="gray", va="bottom", fontsize=8)
plt.tight_layout(); savefig("fig02_curvas_treino.png"); plt.show()

## 4. Avaliação quantitativa

**Escolha do limiar de confiança.** O limiar 0,5 é só um padrão. Na triagem, **deixar passar uma fissura (falso negativo) custa mais** do que pedir uma inspeção a mais (falso positivo). Por isso o limiar é escolhido **na validação** maximizando o F-beta com beta = 2 (recall pesa 2× mais que precisão). O teste não participa dessa escolha.

In [ ]:
def metrics_at(y, p, t):
    yhat = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    return dict(thr=float(t), acc=accuracy_score(y, yhat), precision=precision_score(y, yhat, zero_division=0),
                recall=recall_score(y, yhat, zero_division=0), f1=f1_score(y, yhat, zero_division=0),
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))

def choose_threshold(y, p, beta=2.0):
    pr, rc, th = precision_recall_curve(y, p)
    pr, rc = pr[:-1], rc[:-1]
    fb = (1 + beta**2) * pr * rc / np.maximum(beta**2 * pr + rc, 1e-9)
    idx = np.where(fb >= fb.max() - 1e-9)[0]            # em caso de empate, usa a mediana (mais estável)
    return float(np.clip(np.median(th[idx]), 0.05, 0.95))

p_val, y_val = predict_probs(model, val_dl)
p_te, y_te = predict_probs(model, test_dl)
THR = choose_threshold(y_val, p_val, beta=2.0)

m05 = metrics_at(y_te, p_te, 0.5)
mthr = metrics_at(y_te, p_te, THR)
auc = roc_auc_score(y_te, p_te)
R["metrics"] = dict(thr=THR, test_05=m05, test_thr=mthr, auc=float(auc), val_thr=metrics_at(y_val, p_val, THR))

tab = pd.DataFrame({"limiar 0,5": m05, f"limiar escolhido ({THR:.2f})": mthr}).loc[["acc", "precision", "recall", "f1"]]
tab.index = ["acurácia", "precisão", "recall", "F1"]
print(f"Limiar escolhido na validação (F-beta, beta=2): {THR:.3f}   |   AUC no teste: {auc:.4f}\n")
print(tab.round(4).to_string())
print("\nRelatório por classe no teste (limiar escolhido):")
print(classification_report(y_te, (p_te >= THR).astype(int), target_names=["sem fissura", "com fissura"], digits=4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, m, title in [(axes[0], m05, "Matriz de confusão · limiar 0,5"), (axes[1], mthr, f"Matriz de confusão · limiar {THR:.2f}")]:
    cm = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=16, color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], ["prev. sem", "prev. com"]); ax.set_yticks([0, 1], ["real sem", "real com"]); ax.set_title(title)
pr, rc, th = precision_recall_curve(y_te, p_te)
axes[2].plot(rc, pr); axes[2].scatter([m05["recall"]], [m05["precision"]], label="limiar 0,5", zorder=3)
axes[2].scatter([mthr["recall"]], [mthr["precision"]], label=f"limiar {THR:.2f}", zorder=3, marker="s")
axes[2].set_xlabel("recall"); axes[2].set_ylabel("precisão"); axes[2].set_title("Curva precisão × recall (teste)"); axes[2].legend(); axes[2].grid(alpha=.3)
plt.tight_layout(); savefig("fig03_matriz_confusao_pr.png"); plt.show()

### 4.1 Substituindo valores nas fórmulas (com os números reais do teste)

As métricas são só contagens da matriz de confusão (TP, FP, FN, TN) combinadas por fórmulas. A célula abaixo substitui os valores nas fórmulas para você ver de onde sai cada número.

In [ ]:
m = mthr
tp, fp, fn, tn = m["tp"], m["fp"], m["fn"], m["tn"]
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}   (limiar {THR:.2f})\n")
print(f"Precisão = TP / (TP + FP) = {tp} / ({tp} + {fp}) = {tp/max(tp+fp,1):.4f}   <- das que o modelo apontou como fissura, quantas eram")
print(f"Recall   = TP / (TP + FN) = {tp} / ({tp} + {fn}) = {tp/max(tp+fn,1):.4f}   <- das fissuras reais, quantas o modelo achou")
pr_, rc_ = tp/max(tp+fp,1), tp/max(tp+fn,1)
print(f"F1       = 2·P·R / (P + R) = 2·{pr_:.4f}·{rc_:.4f} / ({pr_:.4f} + {rc_:.4f}) = {2*pr_*rc_/max(pr_+rc_,1e-9):.4f}")
b2 = 4
print(f"F-beta(2)= 5·P·R / (4·P + R) = 5·{pr_:.4f}·{rc_:.4f} / (4·{pr_:.4f} + {rc_:.4f}) = {5*pr_*rc_/max(b2*pr_+rc_,1e-9):.4f}   <- o critério usado para escolher o limiar")

### 4.2 Explorando o limiar (interativo)

Mova o controle e veja a matriz de confusão e as métricas mudarem **na validação** (usada para decidir; o teste fica reservado). Limiar baixo = mais alertas e mais recall; limiar alto = menos alertas e mais precisão.

In [ ]:
try:
    import ipywidgets as W
    from IPython.display import display

    def explore(limiar=0.5):
        mm = metrics_at(y_val, p_val, limiar)
        cm = np.array([[mm["tn"], mm["fp"]], [mm["fn"], mm["tp"]]])
        fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
        ax[0].imshow(cm, cmap="Blues")
        for i in range(2):
            for j in range(2):
                ax[0].text(j, i, cm[i, j], ha="center", va="center", fontsize=15, color="white" if cm[i, j] > cm.max() / 2 else "black")
        ax[0].set_xticks([0, 1], ["prev. sem", "prev. com"]); ax[0].set_yticks([0, 1], ["real sem", "real com"])
        ax[0].set_title(f"Validação · limiar {limiar:.2f}")
        ax[1].bar(["precisão", "recall", "F1"], [mm["precision"], mm["recall"], mm["f1"]], color=["tab:blue", "tab:red", "tab:green"])
        ax[1].set_ylim(0, 1.05)
        for i, v in enumerate([mm["precision"], mm["recall"], mm["f1"]]):
            ax[1].text(i, v + .02, f"{v:.3f}", ha="center")
        plt.tight_layout(); plt.show()
        print(f"FP (alertas desnecessários) = {mm['fp']}   |   FN (fissuras perdidas) = {mm['fn']}")

    W.interact(explore, limiar=W.FloatSlider(value=float(THR), min=0.02, max=0.98, step=0.01, description="limiar", continuous_update=False, layout=W.Layout(width="520px")))
except Exception as e:
    print("ipywidgets indisponível aqui; use a célula anterior para testar limiares manualmente. Detalhe:", e)

## 5. Análise de acertos e erros

Três ferramentas: (a) as imagens erradas com maior confiança, (b) um mapa **Grad-CAM** (onde o modelo "olhou") sobre cada uma, (c) estatísticas simples de brilho, contraste e textura por tipo de resultado (TP/TN/FP/FN) para testar hipóteses sobre sombras, iluminação e textura.

In [ ]:
def cam_overlay(model, arch, path):
    """Grad-CAM: sobrepõe à imagem as regiões que mais aumentaram a pontuação de 'fissura'."""
    img = Image.open(path).convert("RGB").resize((S, S))
    x = tfm_eval(img).unsqueeze(0).to(DEVICE)
    layer = model.features[-1] if "mobilenet" in arch else model.layer4[-1]
    store = {}
    def fwd(mod, inp, out):
        store["a"] = out.detach()
        out.register_hook(lambda g: store.__setitem__("g", g.detach()))
    h = layer.register_forward_hook(fwd)
    model.eval(); model.zero_grad()
    with torch.enable_grad():
        model(x).sum().backward()
    h.remove()
    w = store["g"].mean((2, 3), keepdim=True)
    cam = torch.relu((w * store["a"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=(S, S), mode="bilinear", align_corners=False)[0, 0].cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    heat = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)[:, :, ::-1]
    return np.array(img), (0.55 * np.array(img) + 0.45 * heat).astype(np.uint8)

df_te["prob"] = p_te
df_te["pred"] = (p_te >= THR).astype(int)
df_te["resultado"] = np.select(
    [(df_te.label == 1) & (df_te.pred == 1), (df_te.label == 0) & (df_te.pred == 0),
     (df_te.label == 0) & (df_te.pred == 1), (df_te.label == 1) & (df_te.pred == 0)],
    ["TP (acerto: fissura)", "TN (acerto: sem fissura)", "FP (alarme falso)", "FN (fissura perdida)"])

def show_cases(sub, title, n=6):
    sub = sub.head(n)
    if sub.empty:
        print(f"{title}: nenhum caso no conjunto de teste."); return
    fig, axes = plt.subplots(2, len(sub), figsize=(2.6 * len(sub), 5.6), squeeze=False)
    for j, (_, r) in enumerate(sub.iterrows()):
        orig, over = cam_overlay(model, MAIN_ARCH, r.path)
        axes[0, j].imshow(orig); axes[0, j].set_title(f"p(fissura)={r.prob:.2f}", fontsize=9)
        axes[1, j].imshow(over)
        for a in axes[:, j]: a.axis("off")
    axes[0, 0].text(-0.05, 0.5, "original", transform=axes[0, 0].transAxes, rotation=90, ha="right", va="center")
    axes[1, 0].text(-0.05, 0.5, "Grad-CAM", transform=axes[1, 0].transAxes, rotation=90, ha="right", va="center")
    plt.suptitle(title); plt.tight_layout(); savefig("fig_" + title.split()[0].lower().strip(":") + ".png"); plt.show()

fp_cases = df_te[df_te.resultado.str.startswith("FP")].sort_values("prob", ascending=False)
fn_cases = df_te[df_te.resultado.str.startswith("FN")].sort_values("prob", ascending=True)
tp_cases = df_te[df_te.resultado.str.startswith("TP")].sort_values("prob", ascending=False)
print(f"Erros no teste: {len(fp_cases)} falsos positivos, {len(fn_cases)} falsos negativos")
show_cases(tp_cases, "Acertos (fissura detectada, maior confiança)")
show_cases(fp_cases, "FP: alarmes falsos de maior confiança")
show_cases(fn_cases, "FN: fissuras perdidas (menor confiança)")

In [ ]:
def img_stats(path):
    g = np.array(Image.open(path).convert("L"), dtype=np.float32)
    return g.mean(), g.std(), cv2.Laplacian(g, cv2.CV_32F).var()

st = np.array([img_stats(p) for p in df_te.path])
df_te["brilho"], df_te["contraste"], df_te["textura"] = st[:, 0], st[:, 1], st[:, 2]
err_tab = df_te.groupby("resultado").agg(n=("prob", "size"), brilho_medio=("brilho", "mean"),
                                         contraste_medio=("contraste", "mean"), textura_medio=("textura", "mean"),
                                         prob_media=("prob", "mean")).round(2)
print("Brilho = média de cinza (0-255) · contraste = desvio do cinza · textura = variância do Laplaciano (bordas/rugosidade)\n")
print(err_tab.to_string())
R["error_table"] = err_tab.reset_index().to_dict(orient="records")
R["n_fp"], R["n_fn"] = int(len(fp_cases)), int(len(fn_cases))

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
order = ["TN (acerto: sem fissura)", "TP (acerto: fissura)", "FP (alarme falso)", "FN (fissura perdida)"]
for a, col, ttl in zip(ax, ["brilho", "contraste", "textura"], ["Brilho médio", "Contraste", "Textura (Laplaciano)"]):
    data = [df_te[df_te.resultado == o][col].values for o in order]
    a.boxplot([d if len(d) else [np.nan] for d in data]); a.set_xticklabels(["TN", "TP", "FP", "FN"]); a.set_title(ttl)
plt.tight_layout(); savefig("fig04_estatisticas_erros.png"); plt.show()

### 5.1 Leitura dos resultados

Preencha depois de olhar as figuras e a tabela acima (o texto abaixo só pode ser escrito com o resultado do **seu** treino):

- **Falsos positivos:** o que o Grad-CAM destaca? Juntas, bordas de formas, riscos, sombras longas, manchas, texturas rugosas?
- **Falsos negativos:** fissuras muito finas, pouco contraste com a superfície ou cobertas por sujeira/obstáculos?
- **Estatísticas:** os erros têm brilho, contraste ou textura diferentes dos acertos? (por exemplo, FP com textura/contraste altos sugerem confusão com rugosidade e sombras.)
- **Limitação do teste:** os recortes vêm de um único campus e de superfícies parecidas; o desempenho em outras cidades, câmeras e iluminações não está medido aqui.

## 6. Comparação de arquiteturas (para justificar a escolha)

Treinamos a ResNet18 com exatamente o mesmo pipeline e medimos o tamanho/latência de três arquiteturas. A MobileNetV3-Large entra só na comparação de custo (pesos aleatórios, o que não altera a latência).

In [ ]:
CMP = {}
pv, yv = p_val, y_val
CMP[MAIN_ARCH] = dict(f1_test=mthr["f1"], recall_test=mthr["recall"], auc=float(auc), trained=True)

if CFG["compare_arch"]:
    model_r, hist_r, t_r, _ = train_model("resnet18")
    pv_r, yv_r = predict_probs(model_r, val_dl)
    pt_r, yt_r = predict_probs(model_r, test_dl)
    thr_r = choose_threshold(yv_r, pv_r)
    mr = metrics_at(yt_r, pt_r, thr_r)
    CMP["resnet18"] = dict(f1_test=mr["f1"], recall_test=mr["recall"], auc=float(roc_auc_score(yt_r, pt_r)), trained=True)
    del model_r

print({k: {kk: (round(vv, 4) if isinstance(vv, float) else vv) for kk, vv in v.items()} for k, v in CMP.items()})

## 7. Desempenho: latência e FPS

Medição com *warm-up* (descartando as primeiras execuções), tensores 3×224×224:

- **Latência por imagem:** lote = 1, média e percentil 95 em ms (importante para resposta imediata);
- **Taxa de processamento:** lote = 32, imagens por segundo (importante para varrer vídeo/sequências);
- medida na **CPU** (cenário de implantação mais provável) e na **GPU**, quando existir.

In [ ]:
@torch.no_grad()
def bench(model_, device, bs=1, size=S, warm=20, iters=100):
    m_ = copy.deepcopy(model_).to(device).eval()
    x = torch.randn(bs, 3, size, size, device=device)
    sync = (lambda: torch.cuda.synchronize()) if device == "cuda" else (lambda: None)
    for _ in range(warm): m_(x)
    sync(); ts = []
    for _ in range(iters):
        t0 = time.perf_counter(); m_(x); sync(); ts.append((time.perf_counter() - t0) * 1000)
    return np.array(ts)

devices = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
rows = []
for arch in ["mobilenet_v3_small", "mobilenet_v3_large", "resnet18"]:
    mdl = model if arch == MAIN_ARCH else build_model(arch, pretrained=False)
    npar = sum(p.numel() for p in mdl.parameters()) / 1e6
    for dev in devices:
        t1 = bench(mdl, dev, bs=1, iters=100)
        tb = bench(mdl, dev, bs=32, iters=20, warm=5)
        rows.append(dict(arquitetura=arch, dispositivo=dev, params_M=round(npar, 2),
                         lat_media_ms=round(t1.mean(), 2), lat_p95_ms=round(np.percentile(t1, 95), 2),
                         imgs_por_s_lote32=round(32 / (tb.mean() / 1000), 1),
                         F1_teste=round(CMP[arch]["f1_test"], 4) if arch in CMP else None))
bench_df = pd.DataFrame(rows)
print(f"Equipamento: CPU = {CPU_NAME} ({N_CPU} núcleos lógicos) | GPU = {GPU_NAME or 'nenhuma'} | resolução de entrada = {S}x{S} | torch threads = {torch.get_num_threads()}\n")
print(bench_df.to_string(index=False))
R["bench"] = bench_df.to_dict(orient="records")
R["hw"] = dict(cpu=CPU_NAME, gpu=GPU_NAME, n_cpu=N_CPU, img_size=S, threads=torch.get_num_threads())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
cpu = bench_df[bench_df.dispositivo == "cpu"]
ax[0].bar(cpu.arquitetura, cpu.lat_media_ms, color=["tab:green", "tab:blue", "tab:gray"]); ax[0].set_title("Latência média por imagem · CPU (ms)")
ax[1].bar(cpu.arquitetura, cpu.imgs_por_s_lote32, color=["tab:green", "tab:blue", "tab:gray"]); ax[1].set_title("Imagens/s · CPU · lote 32")
for a in ax: a.tick_params(axis="x", rotation=15)
plt.tight_layout(); savefig("fig05_latencia.png"); plt.show()

## 8. Integração ao protótipo CitizenTech

Salvamos o modelo (com o limiar escolhido) e geramos dois arquivos que formam o módulo:

- `citizentech_crack_module.py`: classe `CrackDetector` com `predict_image` (uma imagem → classe + confiança), `scan_frame` (varre um quadro em blocos de 227 px e devolve os trechos suspeitos, funcionando como localização aproximada por caixas) e `process_video` (vídeo → vídeo anotado + estatísticas de FPS);
- `app_citizentech.py`: tela Streamlit com envio de imagem e de vídeo.

In [ ]:
ckpt_path = OUT_DIR / "crack_model.pt"
torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
            "arch": MAIN_ARCH, "threshold": float(THR), "img_size": int(S)}, ckpt_path)
print("Checkpoint salvo:", ckpt_path, f"({ckpt_path.stat().st_size/1e6:.1f} MB)")

In [ ]:
%%writefile citizentech_crack_module.py
"""Módulo de visão computacional do CitizenTech: triagem de fissuras em concreto.

Apoia a TRIAGEM de locais para inspeção. A presença de fissura na imagem, isoladamente,
não permite determinar a gravidade do dano nem a segurança estrutural da edificação.
"""
import subprocess
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]


def _build(arch):
    if arch == "mobilenet_v3_small":
        m = models.mobilenet_v3_small(weights=None)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, 1)
    elif arch == "mobilenet_v3_large":
        m = models.mobilenet_v3_large(weights=None)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, 1)
    elif arch == "resnet18":
        m = models.resnet18(weights=None)
        m.fc = nn.Linear(m.fc.in_features, 1)
    else:
        raise ValueError(arch)
    return m


class CrackDetector:
    def __init__(self, ckpt_path, device=None, threshold=None):
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        ck = torch.load(str(ckpt_path), map_location="cpu")
        self.arch, self.size = ck["arch"], int(ck["img_size"])
        self.threshold = float(threshold if threshold is not None else ck["threshold"])
        self.model = _build(self.arch)
        self.model.load_state_dict(ck["state_dict"])
        self.model.to(self.device).eval()
        self.mean = torch.tensor(MEAN, device=self.device).view(1, 3, 1, 1)
        self.std = torch.tensor(STD, device=self.device).view(1, 3, 1, 1)

    def _sync(self):
        if self.device.type == "cuda":
            torch.cuda.synchronize()

    @torch.no_grad()
    def predict_rgb_batch(self, arrays, batch_size=64):
        """arrays: lista de imagens RGB uint8 (H, W, 3). Devolve a probabilidade de fissura de cada uma."""
        out = []
        for i in range(0, len(arrays), batch_size):
            chunk = [cv2.resize(np.ascontiguousarray(a), (self.size, self.size)) for a in arrays[i:i + batch_size]]
            x = torch.from_numpy(np.stack(chunk)).to(self.device).permute(0, 3, 1, 2).float() / 255.0
            x = (x - self.mean) / self.std
            out.append(torch.sigmoid(self.model(x).squeeze(1)).cpu().numpy())
        return np.concatenate(out) if out else np.zeros(0, dtype=np.float32)

    def predict_image(self, img):
        """img: caminho, PIL.Image ou array RGB. Devolve classe, confiança e se deve ser inspecionada."""
        if isinstance(img, (str, Path)):
            img = Image.open(img)
        arr = np.array(img.convert("RGB")) if isinstance(img, Image.Image) else np.asarray(img)
        p = float(self.predict_rgb_batch([arr])[0])
        flag = p >= self.threshold
        return {"prob_fissura": p, "classe": "com fissura" if flag else "sem fissura",
                "confianca": p if flag else 1.0 - p, "inspecionar": bool(flag), "limiar": self.threshold}

    def scan_frame(self, frame_bgr, tile=227, stride=None):
        """Varre o quadro em blocos tile x tile e classifica cada bloco (localização aproximada)."""
        h, w = frame_bgr.shape[:2]
        tile = int(min(tile, h, w))
        stride = int(stride or tile)
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        pos = [(x, y) for y in range(0, h - tile + 1, stride) for x in range(0, w - tile + 1, stride)]
        probs = self.predict_rgb_batch([rgb[y:y + tile, x:x + tile] for x, y in pos])
        return [{"x": x, "y": y, "size": tile, "prob": float(p), "flag": bool(p >= self.threshold)}
                for (x, y), p in zip(pos, probs)]

    def annotate_frame(self, frame_bgr, dets, extra_text=""):
        out = frame_bgr.copy()
        flagged = [d for d in dets if d["flag"]]
        for d in flagged:
            x, y, s = d["x"], d["y"], d["size"]
            cv2.rectangle(out, (x, y), (x + s, y + s), (0, 0, 255), 3)
            cv2.putText(out, f"{d['prob']:.0%}", (x + 6, y + 24), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 0, 255), 2, cv2.LINE_AA)
        bar = f"Triagem: {len(flagged)}/{len(dets)} trechos para inspecao | limiar={self.threshold:.2f} {extra_text}"
        cv2.rectangle(out, (0, 0), (out.shape[1], 30), (0, 0, 0), -1)
        cv2.putText(out, bar, (8, 21), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
        return out

    def process_video(self, src, dst, tile=227, frame_stride=1, max_frames=None, fps_out=None):
        """Processa um vídeo (ou sequência) e grava o vídeo anotado. Devolve estatísticas de desempenho."""
        cap = cv2.VideoCapture(str(src))
        if not cap.isOpened():
            raise IOError(f"não foi possível abrir {src}")
        fps_in = cap.get(cv2.CAP_PROP_FPS) or 10.0
        W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        writer = cv2.VideoWriter(str(dst), cv2.VideoWriter_fourcc(*"mp4v"), fps_out or fps_in, (W, H))
        i, n_proc, t_inf, last, log = 0, 0, 0.0, [], []
        t_all = time.perf_counter()
        while True:
            ok, fr = cap.read()
            if not ok or (max_frames is not None and i >= max_frames):
                break
            if i % frame_stride == 0:
                t0 = time.perf_counter()
                last = self.scan_frame(fr, tile)
                self._sync()
                t_inf += time.perf_counter() - t0
                n_proc += 1
            log.append({"frame": i, "blocos": len(last), "suspeitos": sum(d["flag"] for d in last),
                        "prob_max": max([d["prob"] for d in last], default=0.0)})
            writer.write(self.annotate_frame(fr, last))
            i += 1
        cap.release()
        writer.release()
        total = time.perf_counter() - t_all
        return {"frames": i, "frames_processados": n_proc, "tempo_total_s": total,
                "fps_pipeline": i / max(total, 1e-9),
                "ms_inferencia_por_quadro": 1000 * t_inf / max(n_proc, 1),
                "fps_inferencia": n_proc / max(t_inf, 1e-9),
                "resolucao": f"{W}x{H}", "dispositivo": str(self.device), "log": log}


def reencode_h264(src, dst):
    """Converte o mp4 para H.264 (toca em navegadores). Precisa do ffmpeg; devolve True se deu certo."""
    try:
        r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(src), "-vcodec", "libx264",
                            "-pix_fmt", "yuv420p", str(dst)])
        return r.returncode == 0
    except FileNotFoundError:
        return False

In [ ]:
%%writefile app_citizentech.py
import tempfile
from pathlib import Path

import numpy as np
import streamlit as st
from PIL import Image

from citizentech_crack_module import CrackDetector, reencode_h264

st.set_page_config(page_title="CitizenTech · Inspeção visual", layout="wide")
st.title("CitizenTech · Triagem visual de fissuras em concreto")
st.caption("Apoio à triagem de locais para inspeção. A presença de fissura na imagem, isoladamente, "
           "não determina a gravidade do dano nem a segurança estrutural.")

ckpt = st.sidebar.text_input("Checkpoint do modelo", "crack_model.pt")
if not Path(ckpt).exists():
    st.error(f"Arquivo '{ckpt}' não encontrado. Coloque o crack_model.pt na mesma pasta do app.")
    st.stop()


@st.cache_resource
def load(path):
    return CrackDetector(path)


det = load(ckpt)
det.threshold = st.sidebar.slider("Limiar de confiança", 0.05, 0.95, float(det.threshold), 0.01)
st.sidebar.write(f"Modelo: {det.arch} · dispositivo: {det.device}")

tab_img, tab_vid = st.tabs(["Imagem", "Vídeo / sequência"])

with tab_img:
    up = st.file_uploader("Envie uma imagem", type=["jpg", "jpeg", "png"], key="img")
    if up:
        img = Image.open(up).convert("RGB")
        res = det.predict_image(img)
        c1, c2 = st.columns([2, 1])
        arr = np.array(img)
        if min(arr.shape[:2]) > 300:       # imagem grande: varre em blocos e marca os trechos suspeitos
            bgr = arr[:, :, ::-1].copy()
            dets = det.scan_frame(bgr)
            c1.image(det.annotate_frame(bgr, dets)[:, :, ::-1], caption="Trechos suspeitos (caixas vermelhas)")
            c2.metric("Trechos para inspeção", f"{sum(d['flag'] for d in dets)} de {len(dets)}")
            c2.metric("Maior confiança", f"{max(d['prob'] for d in dets):.1%}")
        else:
            c1.image(img, caption="Imagem enviada")
            c2.metric("Classe", res["classe"])
            c2.metric("Confiança", f"{res['confianca']:.1%}")
            c2.metric("p(fissura)", f"{res['prob_fissura']:.1%}")
            (c2.warning if res["inspecionar"] else c2.success)(
                "Possível necessidade de inspeção" if res["inspecionar"] else "Sem indício de fissura")

with tab_vid:
    upv = st.file_uploader("Envie um vídeo curto", type=["mp4", "avi", "mov"], key="vid")
    stride = st.number_input("Processar 1 a cada N quadros", 1, 30, 1)
    if upv and st.button("Processar vídeo"):
        with tempfile.TemporaryDirectory() as tmp:
            src = Path(tmp) / "in.mp4"
            src.write_bytes(upv.read())
            raw, web = Path(tmp) / "out_raw.mp4", Path(tmp) / "out_web.mp4"
            with st.spinner("Processando..."):
                stats = det.process_video(src, raw, frame_stride=int(stride))
            final = web if reencode_h264(raw, web) else raw
            st.video(final.read_bytes())
        c = st.columns(4)
        c[0].metric("Quadros", stats["frames"])
        c[1].metric("FPS do pipeline", f"{stats['fps_pipeline']:.1f}")
        c[2].metric("ms de inferência/quadro", f"{stats['ms_inferencia_por_quadro']:.0f}")
        c[3].metric("Resolução", stats["resolucao"])
        st.line_chart({"trechos suspeitos por quadro": [r["suspeitos"] for r in stats["log"]]})

### 8.1 Demonstração: imagens

In [ ]:
sys.path.insert(0, os.getcwd())
import importlib, citizentech_crack_module as cc
importlib.reload(cc)

det = cc.CrackDetector(ckpt_path)
print(f"Detector carregado ({det.arch}) em {det.device} · limiar = {det.threshold:.2f}\n")

demo = pd.concat([df_te[df_te.label == 1].sample(4, random_state=1), df_te[df_te.label == 0].sample(4, random_state=1)])
fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for ax, (_, r) in zip(axes.ravel(), demo.iterrows()):
    res = det.predict_image(r.path)
    ax.imshow(Image.open(r.path).convert("RGB")); ax.axis("off")
    ax.set_title(f"real: {'COM' if r.label else 'SEM'} fissura\nprevisto: {res['classe']} ({res['confianca']:.1%})",
                 fontsize=10, color="tab:red" if res["inspecionar"] else "tab:green")
plt.tight_layout(); savefig("fig06_demo_imagens.png"); plt.show()

# latência de ponta a ponta (arquivo -> decodificação -> pré-processamento -> modelo -> resultado)
paths = list(df_te.path.sample(min(100, len(df_te)), random_state=0))
det.predict_image(paths[0])
t0 = time.perf_counter()
for p in paths: det.predict_image(p)
e2e_ms = 1000 * (time.perf_counter() - t0) / len(paths)
print(f"Latência de ponta a ponta (disco -> resultado, uma imagem por vez, {det.device}): {e2e_ms:.1f} ms/imagem")
R["e2e_ms"] = e2e_ms

### 8.2 Demonstração: vídeo (simulação de monitoramento urbano)

Sem um vídeo real à mão, simulamos uma passagem de veículo: montamos um "corredor" com recortes do **conjunto de teste** (cerca de 35% com fissura) e deslizamos uma janela de 1135×681 px sobre ele. O módulo varre cada quadro em blocos de 227 px e marca em vermelho os trechos suspeitos com a confiança do modelo.

Como a janela desliza em passos de 57 px, os blocos da varredura não coincidem com os recortes originais: parte dos blocos pega metade de dois recortes. Isso deixa o cenário mais próximo da realidade (a fissura nunca está alinhada à grade), mas significa que **este vídeo é demonstração de funcionamento e velocidade, não uma medida de acurácia**.

Para usar um vídeo seu: `det.process_video("meu_video.mp4", "saida.mp4")`.

In [ ]:
TILE, ROWS, COLS, WIN_COLS, STEP = 227, 3, 24, 5, 57
rs = np.random.RandomState(7)
pos_p = df_te[df_te.label == 1].path.tolist(); neg_p = df_te[df_te.label == 0].path.tolist()
corridor = np.zeros((ROWS * TILE, COLS * TILE, 3), np.uint8)
for r in range(ROWS):
    for c in range(COLS):
        pth = rs.choice(pos_p) if rs.rand() < 0.35 else rs.choice(neg_p)
        corridor[r*TILE:(r+1)*TILE, c*TILE:(c+1)*TILE] = np.array(Image.open(pth).convert("RGB").resize((TILE, TILE)))

demo_in = OUT_DIR / "demo_video_entrada.mp4"
Wd, Hd = WIN_COLS * TILE, ROWS * TILE
vw = cv2.VideoWriter(str(demo_in), cv2.VideoWriter_fourcc(*"mp4v"), 8, (Wd, Hd))
for x0 in range(0, corridor.shape[1] - Wd + 1, STEP):
    vw.write(corridor[:, x0:x0 + Wd][:, :, ::-1])
vw.release()

demo_raw, demo_web = OUT_DIR / "demo_video_saida_raw.mp4", OUT_DIR / "demo_video_saida.mp4"
vstats = det.process_video(demo_in, demo_raw)
R["video"] = {k: v for k, v in vstats.items() if k != "log"}
print(f"Quadros: {vstats['frames']} | resolução: {vstats['resolucao']} | dispositivo: {vstats['dispositivo']}")
print(f"FPS do pipeline completo (leitura + inferência + desenho + gravação): {vstats['fps_pipeline']:.1f}")
print(f"Inferência: {vstats['ms_inferencia_por_quadro']:.1f} ms/quadro ({vstats['fps_inferencia']:.1f} quadros/s, 15 blocos por quadro)")

cap = cv2.VideoCapture(str(demo_raw)); picks = [0, vstats["frames"] // 3, 2 * vstats["frames"] // 3]
fig, axes = plt.subplots(1, 3, figsize=(16, 3.4))
for ax, k in zip(axes, picks):
    cap.set(cv2.CAP_PROP_POS_FRAMES, k); ok, fr = cap.read()
    ax.imshow(fr[:, :, ::-1]); ax.axis("off"); ax.set_title(f"quadro {k}")
cap.release(); plt.tight_layout(); savefig("fig07_demo_video_quadros.png"); plt.show()

shown = demo_web if cc.reencode_h264(demo_raw, demo_web) else demo_raw
try:
    from IPython.display import Video, display
    display(Video(str(shown), embed=True, width=720))
except Exception as e:
    print("Vídeo salvo em", shown, "(", e, ")")

susp = [r["suspeitos"] for r in vstats["log"]]
plt.figure(figsize=(10, 2.6)); plt.plot(susp); plt.title("Trechos marcados para inspeção por quadro"); plt.xlabel("quadro"); plt.tight_layout()
savefig("fig08_suspeitos_por_quadro.png"); plt.show()

### 8.3 Rodando o app Streamlit

Para a **gravação do vídeo de demonstração**, o jeito mais simples é rodar o app no seu computador:

1. baixe o `outputs.zip` gerado na última célula deste notebook e extraia;
2. na pasta extraída: `pip install streamlit torch torchvision opencv-python-headless pillow`;
3. `streamlit run app_citizentech.py`;
4. grave a tela mostrando: envio de uma imagem com fissura, uma sem fissura, ajuste do limiar e o processamento de `demo_video_entrada.mp4`.

Rodando dentro do Colab (opcional) é preciso expor a porta: `!streamlit run app_citizentech.py &>/dev/null &` seguido de um túnel (por exemplo `npx localtunnel --port 8501`).

## 9. Justificativa das escolhas

**Arquitetura (MobileNetV3-Small, transferência de aprendizado).**
- *Qualidade:* a tarefa é binária e com padrão visual marcante (linha escura e fina sobre textura contínua); modelos pré-treinados no ImageNet já trazem filtros de borda/textura úteis, então pouco ajuste basta. A comparação da seção 6 mostra se uma rede maior (ResNet18) compra qualidade que valha o custo.
- *Velocidade:* o monitoramento urbano é contínuo e provavelmente roda em equipamento modesto (computador de bordo, servidor da prefeitura sem GPU). A seção 7 mostra latência e imagens/s de cada arquitetura.
- *Dados e tempo:* com ~6 mil imagens e 4 horas, treinar do zero seria arriscado; congelar o backbone na primeira etapa evita destruir os pesos pré-treinados.

**Pré-processamento.** Redimensionar para 224×224 e normalizar como o ImageNet para casar com o pré-treino; aumento de dados de giro e iluminação porque a base não tem aumento e a câmera/luz reais variam. Para vídeo, a varredura em blocos de 227 px mantém a escala em que o modelo foi treinado (reduzir um quadro inteiro para 224 apagaria fissuras finas).

**Limiar de confiança.** Escolhido na validação com F-beta (beta = 2), privilegiando recall: na triagem, perder uma fissura sai mais caro que pedir uma vistoria a mais. O controle da seção 4.2 e o slider do app permitem ajustar esse equilíbrio conforme a capacidade de inspeção do município. Em vídeo, um limiar mais alto reduz alarmes por quadro (menos "pisca-pisca"), mas pode perder fissuras finas.

**Limitações.** (i) base de um único campus; (ii) a divisão por grupos estimados é aproximada; (iii) a localização por blocos é grosseira, não é detecção por caixas delimitadoras; (iv) o sistema não mede largura, extensão nem gravidade da fissura: só sinaliza onde vale a pena inspecionar.

In [ ]:
def fmt(x, nd=4): return f"{x:.{nd}f}"
b = pd.DataFrame(R["bench"])
cpu_main = b[(b.arquitetura == MAIN_ARCH) & (b.dispositivo == "cpu")].iloc[0]
gpu_rows = b[(b.arquitetura == MAIN_ARCH) & (b.dispositivo == "cuda")]
mt, m5 = R["metrics"]["test_thr"], R["metrics"]["test_05"]
et = pd.DataFrame(R["error_table"])
v = R["video"]

L = []
L.append("# Relatório breve — Visão computacional para manutenção urbana (CitizenTech)\n")
L.append("> O sistema apoia a triagem de locais para inspeção. A presença de fissura na imagem, isoladamente, não permite determinar a gravidade do dano nem a segurança estrutural da edificação.\n")
L.append("## 1. Problema e dados")
L.append(f"- Tarefa: classificação binária (com/sem fissura) em concreto. Base: *Concrete Crack Images for Classification* (Mendeley), imagens 227×227.")
L.append(f"- Amostra: {R['data']['sample']} imagens balanceadas (de {R['data']['avail_pos']} com fissura e {R['data']['avail_neg']} sem fissura disponíveis).")
sp = pd.DataFrame(R["split"])
L.append(f"- Divisão sem vazamento: grupos de origem estimados por K-Means sobre características da CNN ({CFG['n_groups']} grupos) e `GroupShuffleSplit`. Treino/validação/teste: {sp.loc['treino','imagens']}/{sp.loc['validação','imagens']}/{sp.loc['teste','imagens']} imagens, nenhum grupo repetido entre conjuntos. Limitação: os nomes dos arquivos não identificam a foto original, então o agrupamento é aproximado.")
L.append(f"- Pré-processamento: redimensionamento {S}×{S}, normalização ImageNet; aumento (giros, brilho/contraste/saturação) só no treino.\n")
L.append("## 2. Modelo")
L.append(f"- MobileNetV3-Small pré-treinada (ImageNet) com saída de 1 neurônio. Treino em duas etapas: cabeça ({CFG['epochs_head']} épocas, backbone congelado) e ajuste fino ({CFG['epochs_ft']} épocas, taxa 10× menor). Melhor época pela F1 de validação: {R['train']['best_epoch']}. Tempo de treino: {R['train']['seconds']:.0f} s em {GPU_NAME or CPU_NAME}.\n")
L.append("## 3. Métricas no conjunto de teste")
L.append("| Limiar | Acurácia | Precisão | Recall | F1 | TP | FP | FN | TN |\n|---|---|---|---|---|---|---|---|---|")
for nome, mm in [("0,50", m5), (f"{R['metrics']['thr']:.2f} (escolhido)", mt)]:
    L.append(f"| {nome} | {fmt(mm['acc'])} | {fmt(mm['precision'])} | {fmt(mm['recall'])} | {fmt(mm['f1'])} | {mm['tp']} | {mm['fp']} | {mm['fn']} | {mm['tn']} |")
L.append(f"\nAUC no teste: {fmt(R['metrics']['auc'])}. Figuras: `fig03_matriz_confusao_pr.png`.\n")
L.append("## 4. Desempenho")
L.append(f"- Equipamento: CPU {R['hw']['cpu']} ({R['hw']['n_cpu']} núcleos lógicos, {R['hw']['threads']} threads)" + (f"; GPU {R['hw']['gpu']}" if R['hw']['gpu'] else "; sem GPU") + f". Resolução de entrada do modelo: {S}×{S}.")
L.append(f"- CPU: latência média {cpu_main.lat_media_ms} ms/imagem (p95 {cpu_main.lat_p95_ms} ms); {cpu_main.imgs_por_s_lote32} imagens/s com lote 32.")
if len(gpu_rows):
    g = gpu_rows.iloc[0]
    L.append(f"- GPU: latência média {g.lat_media_ms} ms/imagem (p95 {g.lat_p95_ms} ms); {g.imgs_por_s_lote32} imagens/s com lote 32.")
L.append(f"- Ponta a ponta (arquivo → resultado, 1 imagem por vez): {R['e2e_ms']:.1f} ms/imagem.")
L.append(f"- Vídeo de demonstração ({v['resolucao']}, 15 blocos de 227 px por quadro, {v['dispositivo']}): {v['fps_pipeline']:.1f} FPS no pipeline completo e {v['ms_inferencia_por_quadro']:.1f} ms de inferência por quadro.\n")
L.append("### Comparação de arquiteturas\n")
L.append(b.to_string(index=False))
L.append("\n## 5. Justificativa de arquitetura, pré-processamento e limiar")
L.append("- Arquitetura: modelo leve pré-treinado, adequado a monitoramento contínuo em equipamento modesto; tarefa binária de padrão visual marcante não exige rede grande. A comparação acima registra o ganho (ou a falta de ganho) de qualidade de uma rede maior frente ao custo de latência.")
L.append(f"- Limiar {R['metrics']['thr']:.2f}: escolhido na validação por F-beta (beta = 2) para privilegiar recall, pois perder uma fissura na triagem custa mais que um alarme extra. O app permite ajustá-lo.")
L.append("- Varredura em blocos de 227 px no vídeo: preserva a escala de treino; reduzir o quadro inteiro apagaria fissuras finas.\n")
L.append("## 6. Análise de erros")
L.append(f"No teste houve {R['n_fp']} falsos positivos e {R['n_fn']} falsos negativos. Médias por tipo de resultado (brilho 0–255, contraste = desvio do cinza, textura = variância do Laplaciano):\n")
L.append(et.to_string(index=False))
L.append("\n> **Complete antes de entregar:** descreva o que as figuras `fig_fp*.png` e `fig_fn*.png` (com Grad-CAM) mostram — sombras, juntas, manchas, texturas rugosas, fissuras muito finas, baixa luminosidade ou obstáculos — e relacione com a tabela acima.\n")
L.append("## 7. Limitações")
L.append("- Base de um único campus; generalização para outras cidades, câmeras e iluminações não medida.\n- Divisão por grupos estimados (aproximada).\n- Localização por blocos (aproximação), não detecção por caixas delimitadoras.\n- O sistema não estima largura, extensão ou gravidade; apenas sinaliza locais para inspeção.")

(OUT_DIR / "relatorio_breve.md").write_text("\n".join(L), encoding="utf-8")
with open(OUT_DIR / "resultados.json", "w") as f:
    json.dump(R, f, indent=2, default=str, ensure_ascii=False)
print("\n".join(L[:12]), "\n...\n[relatório completo em outputs/relatorio_breve.md]")

## 10. Empacotar entregas

In [ ]:
for f in ["citizentech_crack_module.py", "app_citizentech.py"]:
    shutil.copy(f, OUT_DIR / f)
if (OUT_DIR / "demo_video_saida_raw.mp4").exists() and (OUT_DIR / "demo_video_saida.mp4").exists():
    (OUT_DIR / "demo_video_saida_raw.mp4").unlink()          # mantém só a versão H.264
zip_path = shutil.make_archive(str(WORK / "outputs"), "zip", OUT_DIR)
print("Pacote:", zip_path)
print(sorted(p.name for p in OUT_DIR.iterdir()))
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    pass

## Checklist de entregas

- [x] **Notebook executável**: este arquivo (preparação de dados, modelo, avaliação, integração).
- [ ] **Vídeo de demonstração**: gravar a tela com o app Streamlit (seção 8.3): imagem com fissura, sem fissura, ajuste do limiar e processamento de `demo_video_entrada.mp4`.
- [ ] **Relatório breve**: `outputs/relatorio_breve.md` já vem com métricas, latência/FPS, equipamento, justificativas e tabela de erros; falta só o parágrafo de leitura visual dos erros (seção 5.1) e converter para PDF/Word se for pedido.